# LeetCode 1137 — N-th Tribonacci Number bằng Python

**Link đề:** https://leetcode.com/problems/n-th-tribonacci-number/
**Độ khó:** Easy · **Chủ đề:** Math, Dynamic Programming, Memoization

```python
class Solution:
    def tribonacci(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **dãy truy hồi** (recurrence): mỗi số được tính từ các số đứng trước.
2. Viết lời giải **đệ quy ngây thơ** và tự đo được vì sao nó chậm (gọi lặp lại cùng một bài con).
3. Học **memoization** (ghi nhớ kết quả) bằng `dict` và bằng `functools.lru_cache`.
4. Chuyển sang **quy hoạch động bảng (bottom-up DP)** rồi tối ưu bộ nhớ còn **3 biến cuộn** — `O(n)` thời gian, `O(1)` bộ nhớ.
5. Trace từng bước, chứng minh invariant, test đối chiếu nhiều cách, lỗi hay gặp, biến thể và bài tập.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài và ví dụ

Dãy **Tribonacci** định nghĩa:

- `T0 = 0`, `T1 = 1`, `T2 = 1`
- `T(n+3) = T(n) + T(n+1) + T(n+2)` với `n >= 0`

Cho `n`, trả về `T(n)`.

| `n` | Output | Giải thích |
|---:|---:|---|
| `4` | `4` | `T3 = 0+1+1 = 2`, `T4 = 1+1+2 = 4` |
| `25` | `1389537` | tính dần lên |

**Constraints:**
- `0 <= n <= 37`
- Đáp án đảm bảo vừa số nguyên 32-bit (`<= 2^31 - 1`).

Nói gọn: **mỗi số = tổng 3 số liền trước**. Giống Fibonacci (tổng 2 số liền trước) nhưng nhìn lùi 3 bước.

### Tính bằng tay vài số đầu

```text
T0 = 0
T1 = 1
T2 = 1
T3 = T0 + T1 + T2 = 0 + 1 + 1 = 2
T4 = T1 + T2 + T3 = 1 + 1 + 2 = 4
T5 = T2 + T3 + T4 = 1 + 2 + 4 = 7
T6 = T3 + T4 + T5 = 2 + 4 + 7 = 13
T7 = 4 + 7 + 13 = 24
```

Viết lại công thức theo cách dễ dùng hơn: với `n >= 3`, **`T(n) = T(n-1) + T(n-2) + T(n-3)`** (chỉ là dịch chỉ số đi 3).

In [1]:
# Kiểm tra bảng tay bằng một list nhỏ
t = [0, 1, 1]
for i in range(3, 8):
    t.append(t[i - 1] + t[i - 2] + t[i - 3])
print(t)   # [0, 1, 1, 2, 4, 7, 13, 24]

[0, 1, 1, 2, 4, 7, 13, 24]


## 2. Nền tảng Python cần biết

### 2.1. List, `append` và chỉ số âm

- `t = [0, 1, 1]` tạo list 3 phần tử.
- `t.append(x)` thêm `x` vào **cuối** list.
- `t[-1]` là phần tử cuối, `t[-2]` áp chót, `t[-3]` đứng trước nữa — rất tiện để lấy "3 số cuối".

In [2]:
t = [0, 1, 1]
t.append(2)
print(t)                 # [0, 1, 1, 2]
print(t[-1], t[-2], t[-3])   # 2 1 1
t.append(t[-1] + t[-2] + t[-3])
print(t)                 # [0, 1, 1, 2, 4]

[0, 1, 1, 2]
2 1 1
[0, 1, 1, 2, 4]


### 2.2. `range(a, b)` — chạy từ `a` đến `b - 1`

`range(3, n + 1)` sinh `3, 4, ..., n`. Phải có `+ 1` vì `range` **không** lấy đầu mút phải.

In [3]:
n = 6
print(list(range(3, n)))       # thiếu n
print(list(range(3, n + 1)))   # đủ tới n

[3, 4, 5]
[3, 4, 5, 6]


### 2.3. Gán đồng thời (tuple unpacking)

Python cho phép `a, b, c = b, c, a + b + c`. Vế phải được **tính xong hết** (dùng giá trị cũ) rồi mới gán sang vế trái. Nhờ vậy "trượt cửa sổ" 3 số chỉ cần một dòng, không cần biến tạm.

In [4]:
a, b_, c = 0, 1, 1
print("trước:", a, b_, c)
a, b_, c = b_, c, a + b_ + c
print("sau 1 bước:", a, b_, c)   # 1 1 2
a, b_, c = b_, c, a + b_ + c
print("sau 2 bước:", a, b_, c)   # 1 2 4

trước: 0 1 1
sau 1 bước: 1 1 2
sau 2 bước: 1 2 4


### 2.4. Đệ quy (recursion) là gì?

Một hàm **tự gọi chính nó** với đầu vào nhỏ hơn, và có **điểm dừng** (base case) để không gọi mãi. Với Tribonacci:

- base case: `n == 0 → 0`, `n == 1 hoặc 2 → 1`
- bước đệ quy: `T(n) = T(n-1) + T(n-2) + T(n-3)`

Ví dụ nhỏ quen thuộc: giai thừa `fact(n) = n * fact(n-1)`, `fact(0) = 1`.

In [5]:
def fact(n: int) -> int:
    if n == 0:          # base case
        return 1
    return n * fact(n - 1)

print(fact(5))   # 120

120


### 2.5. `dict` để ghi nhớ kết quả

`memo = {}` là từ điển: key → value. `memo[5] = 7` lưu "T5 = 7"; `5 in memo` kiểm tra đã lưu chưa (trung bình `O(1)`).

In [6]:
memo = {}
memo[5] = 7
print(memo)          # {5: 7}
print(5 in memo)     # True
print(6 in memo)     # False
print(memo[5])       # 7

{5: 7}
True
False
7


## 3. Cách 1 — Đệ quy ngây thơ (dịch thẳng công thức)

Cách trực giác nhất: viết đúng y như định nghĩa toán học.

In [7]:
def trib_naive(n: int) -> int:
    if n == 0:
        return 0
    if n <= 2:
        return 1
    return trib_naive(n - 1) + trib_naive(n - 2) + trib_naive(n - 3)

print(trib_naive(4))    # 4
print(trib_naive(10))   # 149

4
149


### Nút thắt: cây gọi hàm phình to theo cấp số nhân

Gọi `T(5)` sẽ gọi `T(4), T(3), T(2)`; `T(4)` lại gọi `T(3), T(2), T(1)`… Cùng một `T(3)` bị tính **nhiều lần**.

```text
                 T5
        /        |        \
      T4         T3        T2
    /  |  \    /  |  \
  T3  T2  T1  T2  T1  T0
 / | \
T2 T1 T0
```

Số lần gọi tăng khoảng `1.84^n` (1.84 là nghiệm của `x³ = x² + x + 1`). Với `n = 37` là hàng chục tỷ lời gọi → quá chậm. Ta đếm thử:

In [8]:
calls = 0

def trib_naive_count(n: int) -> int:
    global calls
    calls += 1
    if n == 0:
        return 0
    if n <= 2:
        return 1
    return trib_naive_count(n - 1) + trib_naive_count(n - 2) + trib_naive_count(n - 3)

for n in [5, 10, 15, 20]:
    calls = 0
    value = trib_naive_count(n)
    print(f"n={n:>2}: T(n)={value:>6}, số lần gọi hàm = {calls}")

n= 5: T(n)=     7, số lần gọi hàm = 13
n=10: T(n)=   149, số lần gọi hàm = 289
n=15: T(n)=  3136, số lần gọi hàm = 6094
n=20: T(n)= 66012, số lần gọi hàm = 128287


Nhận xét: `n` tăng 5 thì số lần gọi tăng ~20 lần. Tới `n = 37` sẽ là khoảng `6 * 10^9` lời gọi — không thể nộp.

**Câu hỏi mấu chốt:** chỉ có `n + 1` giá trị khác nhau `T(0)..T(n)`, vậy tại sao phải tính hàng tỷ lần? → **Ghi nhớ lại** kết quả đã tính.

## 4. Insight — Bài con chồng lấp (overlapping subproblems)

- Mỗi `T(k)` **chỉ có một giá trị duy nhất**, không phụ thuộc ai gọi nó.
- Vậy lần đầu tính xong thì **lưu lại**; lần sau chỉ việc tra.
- Đây là ý tưởng cốt lõi của **Dynamic Programming (DP)**: chia bài lớn thành bài con, bài con lặp lại thì nhớ kết quả.

Có hai hướng:
1. **Top-down (memoization):** giữ nguyên đệ quy, thêm bộ nhớ đệm.
2. **Bottom-up (tabulation):** tính từ nhỏ đến lớn bằng vòng lặp.

## 5. Cách 2 — Đệ quy có ghi nhớ (memoization)

Thêm `memo` (dict). Trước khi tính, kiểm tra `n in memo`; tính xong thì `memo[n] = kết quả`.

In [9]:
def trib_memo(n: int, memo=None) -> int:
    if memo is None:
        memo = {}
    if n == 0:
        return 0
    if n <= 2:
        return 1
    if n in memo:                       # đã tính rồi → tra
        return memo[n]
    memo[n] = trib_memo(n - 1, memo) + trib_memo(n - 2, memo) + trib_memo(n - 3, memo)
    return memo[n]

print(trib_memo(4), trib_memo(25), trib_memo(37))

4 1389537 2082876103


### Vì sao `memo=None` chứ không `memo={}`?

Giá trị mặc định của tham số chỉ được tạo **một lần** khi định nghĩa hàm. Viết `memo={}` thì mọi lần gọi dùng **chung** một dict — chạy vẫn ra đúng ở bài này, nhưng là thói quen nguy hiểm (gọi là *mutable default argument*). Dùng `None` rồi tạo mới bên trong là cách an toàn.

In [10]:
def add_item(x, bag=[]):        # cố tình viết sai để thấy hiện tượng
    bag.append(x)
    return bag

print(add_item(1))   # [1]
print(add_item(2))   # [1, 2]  ← bất ngờ: list cũ vẫn còn

[1]
[1, 2]


### Đếm số lần gọi sau khi có memo

In [11]:
calls = 0

def trib_memo_count(n: int, memo: dict) -> int:
    global calls
    calls += 1
    if n == 0:
        return 0
    if n <= 2:
        return 1
    if n in memo:
        return memo[n]
    memo[n] = trib_memo_count(n - 1, memo) + trib_memo_count(n - 2, memo) + trib_memo_count(n - 3, memo)
    return memo[n]

for n in [5, 10, 20, 37]:
    calls = 0
    v = trib_memo_count(n, {})
    print(f"n={n:>2}: T(n)={v:>11}, số lần gọi = {calls}")

n= 5: T(n)=          7, số lần gọi = 10
n=10: T(n)=        149, số lần gọi = 25
n=20: T(n)=      66012, số lần gọi = 55
n=37: T(n)= 2082876103, số lần gọi = 106


Số lần gọi giờ chỉ khoảng `3n` → **tuyến tính**. Từ hàng tỷ xuống còn hơn 100 lời gọi cho `n = 37`.

### `functools.lru_cache` — memo "có sẵn"

Decorator `@lru_cache(maxsize=None)` (hoặc `@cache` từ Python 3.9) tự động ghi nhớ kết quả theo tham số. Ta viết code như bản ngây thơ mà vẫn nhanh.

In [12]:
from functools import lru_cache

@lru_cache(maxsize=None)
def trib_cache(n: int) -> int:
    if n == 0:
        return 0
    if n <= 2:
        return 1
    return trib_cache(n - 1) + trib_cache(n - 2) + trib_cache(n - 3)

print(trib_cache(37))
print(trib_cache.cache_info())

2082876103
CacheInfo(hits=68, misses=38, maxsize=None, currsize=38)


**Nhược điểm của top-down:** vẫn dùng ngăn xếp đệ quy sâu `n` tầng (Python mặc định giới hạn ~1000 tầng). Với `n <= 37` thì ổn, nhưng nếu `n` lớn sẽ gặp `RecursionError`. Hướng bottom-up tránh được điều đó.

## 6. Cách 3 — DP bảng (bottom-up)

Tạo mảng `dp` dài `n + 1` (hoặc 38 như gợi ý của đề), điền `dp[0], dp[1], dp[2]`, rồi chạy vòng lặp từ 3 lên `n`.

**Quy tắc chuyển trạng thái:** `dp[i] = dp[i-1] + dp[i-2] + dp[i-3]`.

In [13]:
def trib_table(n: int) -> int:
    if n == 0:
        return 0
    if n <= 2:
        return 1
    dp = [0] * (n + 1)
    dp[1] = dp[2] = 1
    for i in range(3, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2] + dp[i - 3]
    return dp[n]

print([trib_table(i) for i in range(10)])
print(trib_table(25))

[0, 1, 1, 2, 4, 7, 13, 24, 44, 81]
1389537


### Giải thích `[0] * (n + 1)` và `dp[1] = dp[2] = 1`

- `[0] * 5` → `[0, 0, 0, 0, 0]`: nhân list với số nguyên = lặp lại phần tử.
- `dp[1] = dp[2] = 1` là **gán chuỗi**: cả hai nhận giá trị 1.
- Vì sao cần `if n <= 2` trước? Nếu `n = 0` thì `dp` dài 1, viết `dp[1]` sẽ `IndexError`.

In [14]:
print([0] * 5)
x = y = 7
print(x, y)
try:
    dp = [0] * 1      # n = 0
    dp[1] = 1
except IndexError as e:
    print("IndexError:", e)

[0, 0, 0, 0, 0]
7 7
IndexError: list assignment index out of range


## 7. Cách 4 — Lời giải chính: 3 biến cuộn (O(1) bộ nhớ)

Quan sát bảng `dp`: để tính `dp[i]` ta chỉ cần **3 ô cuối**. Các ô cũ hơn không bao giờ dùng lại → không cần lưu cả mảng.

Giữ 3 biến:
- `a = T(i-3)`, `b = T(i-2)`, `c = T(i-1)`

Mỗi bước: `a, b, c = b, c, a + b + c` — cửa sổ trượt sang phải một ô, `c` mới chính là `T(i)`.

### Mổ xẻ từng đoạn
1. Xử lý base case: `n == 0 → 0`, `n <= 2 → 1`.
2. Khởi tạo `a, b, c = 0, 1, 1` (tức `T0, T1, T2`).
3. Lặp `n - 2` lần (cho `i = 3..n`), mỗi lần trượt cửa sổ.
4. Trả `c`.

In [15]:
def trib_rolling(n: int) -> int:
    if n == 0:
        return 0
    if n <= 2:
        return 1
    a, b, c = 0, 1, 1           # T0, T1, T2
    for _ in range(3, n + 1):
        a, b, c = b, c, a + b + c
    return c

print(trib_rolling(4), trib_rolling(25), trib_rolling(37))

4 1389537 2082876103


`for _ in ...`: dấu gạch dưới `_` là tên biến quy ước cho "không dùng tới" — ta chỉ cần lặp đủ số lần, không cần giá trị `i`.

## 8. TRACE từng bước với `n = 6`

| bước (i) | a = T(i-3) | b = T(i-2) | c = T(i-1) | c mới = a+b+c = T(i) |
|---:|---:|---:|---:|---:|
| bắt đầu | 0 | 1 | 1 | — |
| 3 | 0 | 1 | 1 | 2 |
| 4 | 1 | 1 | 2 | 4 |
| 5 | 1 | 2 | 4 | 7 |
| 6 | 2 | 4 | 7 | 13 |

Cột `a, b, c` ghi **trước** khi cập nhật ở bước đó. Kết thúc: `c = 13 = T6`.

In [16]:
def trace_trib(n: int) -> int:
    if n == 0:
        print("n = 0 → trả 0 ngay")
        return 0
    if n <= 2:
        print(f"n = {n} → trả 1 ngay")
        return 1
    a, b, c = 0, 1, 1
    print(f"bắt đầu: a={a}, b={b}, c={c}")
    for i in range(3, n + 1):
        new_c = a + b + c
        print(f"i={i}: a={a}, b={b}, c={c} -> T({i}) = {a}+{b}+{c} = {new_c}")
        a, b, c = b, c, new_c
    print(f"kết quả T({n}) = {c}")
    return c

trace_trib(6)
print()
trace_trib(2)

bắt đầu: a=0, b=1, c=1
i=3: a=0, b=1, c=1 -> T(3) = 0+1+1 = 2
i=4: a=1, b=1, c=2 -> T(4) = 1+1+2 = 4
i=5: a=1, b=2, c=4 -> T(5) = 1+2+4 = 7
i=6: a=2, b=4, c=7 -> T(6) = 2+4+7 = 13
kết quả T(6) = 13

n = 2 → trả 1 ngay


1

## 9. Code nộp LeetCode

Bản 3 biến cuộn — gọn, `O(n)` thời gian, `O(1)` bộ nhớ.

In [17]:
class Solution:
    def tribonacci(self, n: int) -> int:
        if n == 0:
            return 0
        if n <= 2:
            return 1
        a, b, c = 0, 1, 1
        for _ in range(3, n + 1):
            a, b, c = b, c, a + b + c
        return c

sol = Solution()
print(sol.tribonacci(4), sol.tribonacci(25))

4 1389537


### Code nộp LeetCode — bản memo (top-down)

Nếu thích tư duy đệ quy, có thể nộp bản `lru_cache`. Đặt hàm phụ bên trong method để cache không lẫn giữa các test.

In [18]:
from functools import lru_cache

class SolutionMemo:
    def tribonacci(self, n: int) -> int:
        @lru_cache(maxsize=None)
        def t(k: int) -> int:
            if k == 0:
                return 0
            if k <= 2:
                return 1
            return t(k - 1) + t(k - 2) + t(k - 3)
        return t(n)

print(SolutionMemo().tribonacci(4), SolutionMemo().tribonacci(37))

4 2082876103


## 10. Vì sao thuật toán đúng? (invariant)

**Invariant của vòng lặp:** ngay trước khi xử lý chỉ số `i` (với `3 <= i <= n`), ta có
`a = T(i-3)`, `b = T(i-2)`, `c = T(i-1)`.

- **Khởi đầu** (`i = 3`): `a, b, c = 0, 1, 1 = T0, T1, T2` ✔.
- **Duy trì:** sau `a, b, c = b, c, a+b+c` ta được `a = T(i-2)`, `b = T(i-1)`, `c = T(i-3)+T(i-2)+T(i-1) = T(i)` — đúng invariant cho `i + 1`.
- **Kết thúc:** vòng lặp chạy xong `i = n`, nên `c = T(n)`.

Các trường hợp `n = 0, 1, 2` được trả trực tiếp theo định nghĩa. Vậy kết quả luôn đúng.

## 11. Độ phức tạp (Time / Space)

Gọi `n` là chỉ số cần tính.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Đệ quy ngây thơ | `O(1.84^n)` | `O(n)` (độ sâu đệ quy) | quá chậm khi `n ≈ 37` |
| Memo (dict / lru_cache) | `O(n)` | `O(n)` (memo + stack) | dễ viết từ công thức |
| DP bảng | `O(n)` | `O(n)` | không đệ quy |
| 3 biến cuộn | `O(n)` | `O(1)` | **lời giải chính** |
| Bảng tính trước 38 số | `O(1)` mỗi truy vấn | `O(38)` | hợp khi gọi rất nhiều lần |

Lưu ý: vì `n <= 37` nên "`O(n)`" ở đây cực nhỏ; phân tích vẫn quan trọng để tư duy khi `n` lớn.

## 12. Bộ test: mẫu đề, biên, đối chiếu nhiều cách

- Mẫu đề: `n = 4 → 4`, `n = 25 → 1389537`.
- Biên: `n = 0, 1, 2, 3`, `n = 37` (lớn nhất).
- Đối chiếu: mọi cách cho cùng kết quả với `n = 0..37` (bản ngây thơ chỉ thử tới 20 để khỏi chậm).

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  ({actual})")

sol = Solution()
check("đề n=4", sol.tribonacci(4), 4)
check("đề n=25", sol.tribonacci(25), 1389537)
check("biên n=0", sol.tribonacci(0), 0)
check("biên n=1", sol.tribonacci(1), 1)
check("biên n=2", sol.tribonacci(2), 1)
check("n=3", sol.tribonacci(3), 2)
check("n=37", sol.tribonacci(37), 2082876103)
assert sol.tribonacci(37) <= 2**31 - 1

đề n=4         PASS  (4)
đề n=25        PASS  (1389537)
biên n=0       PASS  (0)
biên n=1       PASS  (1)
biên n=2       PASS  (1)
n=3            PASS  (2)
n=37           PASS  (2082876103)


In [20]:
for n in range(38):
    expected = trib_table(n)
    assert trib_rolling(n) == expected
    assert trib_memo(n) == expected
    assert trib_cache(n) == expected
    assert Solution().tribonacci(n) == expected
    assert SolutionMemo().tribonacci(n) == expected
    if n <= 20:
        assert trib_naive(n) == expected
print("Mọi cách khớp nhau với n = 0..37. PASS")

Mọi cách khớp nhau với n = 0..37. PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên base case `n = 0` khi dùng bảng
```python
dp = [0] * (n + 1)
dp[1] = dp[2] = 1     # n = 0 hoặc 1 → IndexError
```
**Sửa:** xử lý `n <= 2` trước khi tạo bảng.

### Lỗi 2: Cập nhật tuần tự làm mất giá trị cũ
```python
a = b
b = c
c = a + b + c   # a, b lúc này đã là giá trị MỚI → sai
```
**Sửa:** dùng gán đồng thời `a, b, c = b, c, a + b + c`, hoặc lưu `new_c = a + b + c` trước.

### Lỗi 3: `range(3, n)` thiếu một vòng → trả `T(n-1)`.

### Lỗi 4: Base case sai `T2 = 2` (nhầm với dãy khác) → lệch toàn bộ dãy.

In [21]:
def trib_wrong_update(n: int) -> int:
    a, b, c = 0, 1, 1
    for _ in range(3, n + 1):
        a = b
        b = c
        c = a + b + c          # dùng a, b đã bị ghi đè
    return c

def trib_wrong_range(n: int) -> int:
    a, b, c = 0, 1, 1
    for _ in range(3, n):      # thiếu +1
        a, b, c = b, c, a + b + c
    return c

print("đúng       T(6) =", trib_rolling(6))
print("cập nhật sai T(6) =", trib_wrong_update(6))
print("range sai  T(6) =", trib_wrong_range(6))

đúng       T(6) = 13
cập nhật sai T(6) = 41
range sai  T(6) = 7


## 14. Mẹo nhận dạng pattern

- Đề cho **công thức truy hồi** `f(n)` phụ thuộc vào vài `f` trước → nghĩ ngay tới DP 1 chiều.
- Nếu `f(n)` chỉ phụ thuộc **k** giá trị trước → giữ **k biến cuộn**, bộ nhớ `O(1)`.
- Đệ quy ngây thơ chậm vì **bài con chồng lấp** → memo hoặc bottom-up.
- Các bài cùng họ: Fibonacci (509), Climbing Stairs (70), Min Cost Climbing Stairs (746), House Robber (198).

**Template DP k biến:**
```python
state = base_values           # k giá trị đầu
for i in range(k, n + 1):
    state = state[1:] + (f(state),)
return state[-1]
```

## 15. Biến thể

### 15.1. Dãy "k-bonacci" tổng quát
Mỗi số là tổng `k` số trước. Dùng `collections.deque(maxlen=k)` làm cửa sổ trượt: khi `append` mà đầy, phần tử đầu tự bị đẩy ra.

In [22]:
from collections import deque

def k_bonacci(n: int, k: int, first: list[int]) -> int:
    # first: k giá trị khởi đầu
    if n < k:
        return first[n]
    window = deque(first, maxlen=k)
    s = sum(window)
    for _ in range(k, n + 1):
        new = s
        s += new - window[0]     # cập nhật tổng cửa sổ trong O(1)
        window.append(new)       # phần tử cũ nhất tự bị đẩy ra
    return window[-1]

print([k_bonacci(i, 2, [0, 1]) for i in range(10)])      # Fibonacci
print([k_bonacci(i, 3, [0, 1, 1]) for i in range(10)])   # Tribonacci

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
[0, 1, 1, 2, 4, 7, 13, 24, 44, 81]


### 15.2. Trả về cả dãy `T0..Tn`
Chỉ cần giữ list và `append` — là bản DP bảng, hữu ích khi cần nhiều giá trị.

### 15.3. Nhân ma trận (nâng cao, `O(log n)`)
Vector `[T(i), T(i-1), T(i-2)]` nhân ma trận `M = [[1,1,1],[1,0,0],[0,1,0]]` ra `[T(i+1), T(i), T(i-1)]`. Lũy thừa nhanh `M^k` cho `O(log n)` phép nhân ma trận 3×3. Không cần với `n <= 37`, nhưng là ý tưởng hay khi `n` tới `10^18` (kèm modulo).

In [23]:
def mat_mul(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(3)) for j in range(3)] for i in range(3)]

def mat_pow(M, p):
    R = [[1 if i == j else 0 for j in range(3)] for i in range(3)]   # ma trận đơn vị
    while p > 0:
        if p & 1:
            R = mat_mul(R, M)
        M = mat_mul(M, M)
        p >>= 1
    return R

def trib_matrix(n: int) -> int:
    if n == 0:
        return 0
    if n <= 2:
        return 1
    P = mat_pow([[1, 1, 1], [1, 0, 0], [0, 1, 0]], n - 2)
    # [T(n), T(n-1), T(n-2)] = P * [T2, T1, T0] = P * [1, 1, 0]
    return P[0][0] * 1 + P[0][1] * 1 + P[0][2] * 0

print([trib_matrix(i) for i in range(10)])
assert all(trib_matrix(i) == trib_rolling(i) for i in range(38))
print("Ma trận khớp với 3 biến cuộn. PASS")

[0, 1, 1, 2, 4, 7, 13, 24, 44, 81]
Ma trận khớp với 3 biến cuộn. PASS


### 15.4. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Đệ quy thẳng | dịch công thức | `O(1.84^n)` | `O(n)` | chỉ để hiểu |
| Memo | đệ quy + nhớ | `O(n)` | `O(n)` | đề truy hồi phức tạp, khó viết vòng lặp |
| DP bảng | lặp từ nhỏ lên | `O(n)` | `O(n)` | cần cả dãy |
| 3 biến | chỉ giữ cửa sổ | `O(n)` | `O(1)` | **nộp bài** |
| Ma trận | lũy thừa nhanh | `O(log n)` | `O(1)` | `n` cực lớn |

## 16. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
Không chạy code, hãy tính `T8`, `T9`, `T10`. (Gợi ý: bắt đầu từ `T5 = 7, T6 = 13, T7 = 24`.)

### Bài 2 — Fibonacci bằng 2 biến cuộn
Viết `fib(n)` với `F0 = 0, F1 = 1`, `F(n) = F(n-1) + F(n-2)`, bộ nhớ `O(1)`.

In [24]:
def fib_exercise(n: int) -> int:
    # TODO: xử lý n <= 1, rồi dùng 2 biến cuộn
    pass

# Test gợi ý (bỏ dấu # sau khi làm xong):
# assert [fib_exercise(i) for i in range(8)] == [0, 1, 1, 2, 3, 5, 8, 13]

### Bài 3 — Dãy Tribonacci đầy đủ
Viết `trib_list(n)` trả list `[T0, T1, ..., Tn]`.

In [25]:
def trib_list_exercise(n: int) -> list[int]:
    # TODO: bắt đầu từ [0, 1, 1], append dần, cắt cho đúng độ dài n + 1
    pass

# assert trib_list_exercise(0) == [0]
# assert trib_list_exercise(6) == [0, 1, 1, 2, 4, 7, 13]

### Bài 4 — Climbing Stairs 3 bước
Mỗi lần được leo 1, 2 hoặc 3 bậc. Có bao nhiêu cách leo đúng `n` bậc (`n >= 0`, coi `n = 0` có 1 cách)?

In [26]:
def climb3_exercise(n: int) -> int:
    # TODO: ways(n) = ways(n-1) + ways(n-2) + ways(n-3)
    pass

# assert [climb3_exercise(i) for i in range(7)] == [1, 1, 2, 4, 7, 13, 24]

### Bài 5 — Tribonacci modulo cho `n` lớn
Tính `T(n) % (10**9 + 7)` với `n` tới `10^6` bằng 3 biến cuộn (lấy modulo mỗi bước).

In [27]:
def trib_mod_exercise(n: int, mod: int = 10**9 + 7) -> int:
    # TODO: như 3 biến cuộn, nhưng c mới = (a + b + c) % mod
    pass

# assert trib_mod_exercise(37) == 2082876103 % (10**9 + 7)

### Bài 6 — Mini challenge: chỉ số đầu tiên vượt ngưỡng
Trả về `n` nhỏ nhất sao cho `T(n) > limit`.

In [28]:
def first_index_over_exercise(limit: int) -> int:
    # TODO: chạy cửa sổ 3 biến, đếm chỉ số, dừng khi giá trị > limit
    pass

# assert first_index_over_exercise(0) == 1
# assert first_index_over_exercise(10) == 6     # T6 = 13
# assert first_index_over_exercise(1000) == 14  # T13 = 927, T14 = 1705

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | cộng 3 số liền trước | T8 = T5+T6+T7 | 7+13+24 |
| 2 | chỉ cần 2 số cuối | `a, b = 0, 1` | `a, b = b, a + b` lặp `n` lần, trả `a` |
| 3 | DP bảng | list + `append(t[-1]+t[-2]+t[-3])` | trả `t[:n+1]` |
| 4 | bước cuối là 1, 2 hoặc 3 | giống Tribonacci, khác base | base `1, 1, 2` |
| 5 | modulo không đổi tính cộng | `(x + y) % m` | `a, b, c = b, c, (a+b+c) % mod` |
| 6 | sinh dần dãy | biến `i` song song với `c` | xử lý riêng `limit < 1` |

## 18. Đáp án tham khảo

**Bài 1:** `T8 = 7 + 13 + 24 = 44`, `T9 = 13 + 24 + 44 = 81`, `T10 = 24 + 44 + 81 = 149`.

In [29]:
def fib_answer(n: int) -> int:
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

def trib_list_answer(n: int) -> list[int]:
    t = [0, 1, 1]
    while len(t) <= n:
        t.append(t[-1] + t[-2] + t[-3])
    return t[:n + 1]

def climb3_answer(n: int) -> int:
    a, b, c = 1, 1, 2          # ways(0), ways(1), ways(2)
    if n == 0:
        return a
    if n == 1:
        return b
    for _ in range(3, n + 1):
        a, b, c = b, c, a + b + c
    return c

def trib_mod_answer(n: int, mod: int = 10**9 + 7) -> int:
    if n == 0:
        return 0
    if n <= 2:
        return 1
    a, b, c = 0, 1, 1
    for _ in range(3, n + 1):
        a, b, c = b, c, (a + b + c) % mod
    return c

def first_index_over_answer(limit: int) -> int:
    a, b, c = 0, 1, 1
    if limit < 0:
        return 0
    if limit < 1:
        return 1
    i = 2
    while c <= limit:
        a, b, c = b, c, a + b + c
        i += 1
    return i

assert [trib_rolling(i) for i in (8, 9, 10)] == [44, 81, 149]
assert [fib_answer(i) for i in range(8)] == [0, 1, 1, 2, 3, 5, 8, 13]
assert trib_list_answer(0) == [0]
assert trib_list_answer(6) == [0, 1, 1, 2, 4, 7, 13]
assert [climb3_answer(i) for i in range(7)] == [1, 1, 2, 4, 7, 13, 24]
assert trib_mod_answer(37) == 2082876103 % (10**9 + 7)
assert first_index_over_answer(0) == 1
assert first_index_over_answer(10) == 6
assert first_index_over_answer(1000) == 14
print("Đáp án tham khảo PASS toàn bộ test!")
print("T(10^6) mod 1e9+7 =", trib_mod_answer(10**6))

Đáp án tham khảo PASS toàn bộ test!


T(10^6) mod 1e9+7 = 71313044


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def tribonacci(self, n: int) -> int:
        if n == 0:
            return 0
        if n <= 2:
            return 1
        a, b, c = 0, 1, 1
        for _ in range(3, n + 1):
            a, b, c = b, c, a + b + c
        return c
```

**Checklist trước khi nộp:**
- [ ] Đã trả đúng `T0 = 0`, `T1 = T2 = 1`?
- [ ] Vòng lặp chạy tới **`n + 1`** (bao gồm `n`)?
- [ ] Cập nhật 3 biến bằng **gán đồng thời**, không ghi đè sớm?
- [ ] Test `n = 0, 1, 2, 3, 4, 25, 37`?
- [ ] Biết giải thích vì sao đệ quy ngây thơ chậm và memo/DP sửa thế nào?